# 00 · Data profiling
Missing values, duplicates, column types and date ranges for every raw table,
before any modelling. Decisions taken from these numbers are recorded in
`docs/data_quality.md`.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 200)
RAW = Path("../data/raw")
con = duckdb.connect()
tables = {p.stem: p for p in sorted(RAW.glob("*.csv"))}


def sql(query: str) -> pd.DataFrame:
    return con.sql(query.replace("RAW", str(RAW))).df()

## Columns: types, nulls, distinct values

In [2]:
def profile(name: str, path: Path) -> pd.DataFrame:
    rel = con.read_csv(str(path), header=True)
    df = rel.df()
    return pd.DataFrame(
        {
            "table": name,
            "column": df.columns,
            "dtype": [str(t) for t in rel.dtypes],
            "rows": len(df),
            "nulls": df.isna().sum().values,
            "null_pct": (df.isna().mean() * 100).round(2).values,
            "distinct": df.nunique().values,
        }
    )


summary = pd.concat([profile(n, p) for n, p in tables.items()], ignore_index=True)
summary

,table,column,dtype,rows,nulls,null_pct,distinct
0,olist_customers_dataset,customer_id,VARCHAR,99441,0,0.00,99441
1,olist_customers_dataset,customer_unique_id,VARCHAR,99441,0,0.00,96096
2,olist_customers_dataset,customer_zip_code_prefix,VARCHAR,99441,0,0.00,14994
3,olist_customers_dataset,customer_city,VARCHAR,99441,0,0.00,4119
4,olist_customers_dataset,customer_state,VARCHAR,99441,0,0.00,27
5,olist_geolocation_dataset,geolocation_zip_code_prefix,VARCHAR,1000163,0,0.00,19015
6,olist_geolocation_dataset,geolocation_lat,DOUBLE,1000163,0,0.00,717372
7,olist_geolocation_dataset,geolocation_lng,DOUBLE,1000163,0,0.00,717615
8,olist_geolocation_dataset,geolocation_city,VARCHAR,1000163,0,0.00,8011
9,olist_geolocation_dataset,geolocation_state,VARCHAR,1000163,0,0.00,27


## Full-row duplicates

In [3]:
pd.Series(
    {n: int(con.read_csv(str(p), header=True).df().duplicated().sum()) for n, p in tables.items()},
    name="duplicate_rows",
)

olist_customers_dataset                   0
olist_geolocation_dataset            261831
olist_order_items_dataset                 0
olist_order_payments_dataset              0
olist_order_reviews_dataset               0
olist_orders_dataset                      0
olist_products_dataset                    0
olist_sellers_dataset                     0
product_category_name_translation         0
Name: duplicate_rows, dtype: int64

## Keys and customer identity

In [4]:
checks = {
    "orders: duplicate order_id": "select count(*) - count(distinct order_id) from 'RAW/olist_orders_dataset.csv'",
    "reviews: duplicate review_id": "select count(*) - count(distinct review_id) from 'RAW/olist_order_reviews_dataset.csv'",
    "reviews: duplicate (review_id, order_id)": "select count(*) - count(distinct (review_id, order_id)) from 'RAW/olist_order_reviews_dataset.csv'",
    "orders with >1 review": "select count(*) from (select order_id from 'RAW/olist_order_reviews_dataset.csv' group by 1 having count(*) > 1)",
    "orders without a review": "select count(*) from 'RAW/olist_orders_dataset.csv' o where not exists (select 1 from 'RAW/olist_order_reviews_dataset.csv' r where r.order_id = o.order_id)",
    "customers: duplicate customer_id": "select count(*) - count(distinct customer_id) from 'RAW/olist_customers_dataset.csv'",
    "customer_id values": "select count(distinct customer_id) from 'RAW/olist_customers_dataset.csv'",
    "customer_unique_id values": "select count(distinct customer_unique_id) from 'RAW/olist_customers_dataset.csv'",
    "unique customers with >1 order": """select count(*) from (
          select c.customer_unique_id from 'RAW/olist_orders_dataset.csv' o
          join 'RAW/olist_customers_dataset.csv' c using (customer_id)
          group by 1 having count(*) > 1)""",
    "orders with items from >1 seller": "select count(*) from (select order_id from 'RAW/olist_order_items_dataset.csv' group by 1 having count(distinct seller_id) > 1)",
    "orders without items": "select count(*) from 'RAW/olist_orders_dataset.csv' o where not exists (select 1 from 'RAW/olist_order_items_dataset.csv' i where i.order_id = o.order_id)",
    "orders without payments": "select count(*) from 'RAW/olist_orders_dataset.csv' o where not exists (select 1 from 'RAW/olist_order_payments_dataset.csv' p where p.order_id = o.order_id)",
}
pd.Series({k: sql(q).iat[0, 0] for k, q in checks.items()}, name="value")

orders: duplicate order_id                      0
reviews: duplicate review_id                  814
reviews: duplicate (review_id, order_id)        0
orders with >1 review                         547
orders without a review                       768
customers: duplicate customer_id                0
customer_id values                          99441
customer_unique_id values                   96096
unique customers with >1 order               2997
orders with items from >1 seller             1278
orders without items                          775
orders without payments                         1
Name: value, dtype: int64

## Orders per month (to choose the analysis period) and status mix

In [5]:
sql("""
select date_trunc('month', order_purchase_timestamp::timestamp)::date as month, count(*) as orders
from 'RAW/olist_orders_dataset.csv' group by 1 order by 1
""")

,month,orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [6]:
sql(
    "select order_status, count(*) as orders from 'RAW/olist_orders_dataset.csv' group by 1 order by 2 desc"
)

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## Delivery dates

In [7]:
sql("""
select
  count(*) filter (where order_status = 'delivered') as delivered,
  count(*) filter (where order_status = 'delivered' and order_delivered_customer_date is null) as delivered_no_date,
  count(*) filter (where order_status <> 'delivered' and order_delivered_customer_date is not null) as not_delivered_with_date,
  count(*) filter (where order_delivered_customer_date::timestamp < order_purchase_timestamp::timestamp) as delivered_before_purchase,
  count(*) filter (where order_estimated_delivery_date is null) as no_estimate
from 'RAW/olist_orders_dataset.csv'
""")

,delivered,delivered_no_date,not_delivered_with_date,delivered_before_purchase,no_estimate
0,96478,8,6,0,0


## Payments vs items + freight

In [8]:
sql("""
with i as (select order_id, sum(price + freight_value) as items from 'RAW/olist_order_items_dataset.csv' group by 1),
     p as (select order_id, sum(payment_value) as paid from 'RAW/olist_order_payments_dataset.csv' group by 1)
select count(*) as orders,
       count(*) filter (where abs(items - paid) > 1) as mismatch_gt_1brl,
       round(sum(abs(items - paid)) filter (where abs(items - paid) > 1), 2) as mismatch_brl_total
from i join p using (order_id)
""")

,orders,mismatch_gt_1brl,mismatch_brl_total
0,98665,249,3262.33


In [9]:
sql(
    "select payment_type, count(*) as payments from 'RAW/olist_order_payments_dataset.csv' group by 1 order by 2 desc"
)

,payment_type,payments
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3


## Product categories

In [10]:
sql("""
select count(*) filter (where product_category_name is null) as products_without_category
from 'RAW/olist_products_dataset.csv'
""")

,products_without_category
0,610


In [11]:
sql("""
select distinct p.product_category_name
from 'RAW/olist_products_dataset.csv' p
left join 'RAW/product_category_name_translation.csv' t using (product_category_name)
where t.product_category_name_english is null
""")

,product_category_name
0,NaN
1,pc_gamer
2,portateis_cozinha_e_preparadores_de_alimentos


## Value ranges

In [12]:
sql("""
select min(price) as min_price, max(price) as max_price,
       min(freight_value) as min_freight, max(freight_value) as max_freight,
       count(*) filter (where price <= 0) as non_positive_price
from 'RAW/olist_order_items_dataset.csv'
""")

,min_price,max_price,min_freight,max_freight,non_positive_price
0,0.85,6735.0,0.0,409.68,0


In [13]:
sql(
    "select review_score, count(*) as reviews from 'RAW/olist_order_reviews_dataset.csv' group by 1 order by 1"
)

,review_score,reviews
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328
